# GR00T N1.7-3B on AWS Trainium

This notebook demonstrates the GR00T N1.7-3B → Trainium port:
- Architecture overview
- Load checkpoint and inspect weights
- Compile all subgraphs (skip-if-compiled)
- Load compiled NEFFs
- Run sample inference with dummy inputs
- Benchmark and display results
- Correctness validation

## 1. Setup

In [1]:
# Activate Neuron environment (run in shell, not here)
# source /opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/bin/activate

import os, sys
import torch
import numpy as np

# Add port directory to path
PORT_DIR = '/home/ubuntu/groot-n1-port'
sys.path.insert(0, PORT_DIR)
sys.path.insert(0, os.path.join(PORT_DIR, 'skills', 'scripts'))
sys.path.insert(0, '/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages')

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
try:
    import torch_neuronx
    print('torch_neuronx: OK')
except ImportError:
    print('torch_neuronx: NOT FOUND (not on Trainium?)')

Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
PyTorch: 2.9.1+cu128


torch_neuronx: OK


## 2. Architecture Overview

```
Image (256x256, patch_size=16)
    │
    ▼
[ViT NEFF]             Qwen3-VL ViT: 24 layers, 1024-dim hidden
    │ [64, 2048]        spatial_merge_size=2 → 64 merged tokens
    ▼
[Backbone NEFF] TP=8   16-layer Qwen3-VL-2B LLM (text+image fusion)
    │ [1, 256, 2048]    full sequence: 1 vision_start + 64 image tokens +
    │                   1 vision_end + 16 text + padding = 256 total
    ▼
[VLLN + VL self-attn NEFF] TP=8
    │ [1, 256, 2048]    4-layer self-attn to refine conditioning
    ▼
    ┌── conditioning_tokens ──────────────────────────────────┐
    │                                                          │
    ▼                                                          │
[DiT NEFF x4] TP=8    32-layer flow-matching diffusion transformer
    │  sa_embs [1,41,1536] = state[1,1,1536] + actions[1,40,1536]
    │  cond    [1,256,2048] ← conditioning_tokens
    │  temb    [1,1536]    ← sinusoidal timestep embedding
    │  output  [1,41,1024] → velocity predictions
    ▼
State/Action decoder (CPU, CategorySpecificMLP)
    ▼
Actions [1, 40, 132] BF16   (ACTION_HORIZON=40, action_dim=132)
```

### Subgraph Compiler Flags
All TP=8 NEFFs use `-O1` (no `--model-type=transformer` — avoids
inappropriate causal-LM fusions that degrade cos_sim to ~0.55).

ViT NEFF uses `--auto-cast=matmult --optlevel 3 --model-type=unet-inference`.

## 3. Load Checkpoint and Inspect Weights

In [2]:
import glob
import time
from safetensors.torch import load_file
from config_constants import MODEL_PATH

print(f'Checkpoint path: {MODEL_PATH}')
files = sorted(glob.glob(os.path.join(MODEL_PATH, '*.safetensors')))
print(f'Safetensors shards: {len(files)}')

print('\nLoading checkpoint...')
t0 = time.time()
hf_sd = {}
for f in files:
    hf_sd.update(load_file(f))
print(f'  {len(hf_sd)} tensors in {time.time()-t0:.1f}s')

# Show subgraph key groups
subgraphs = {
    'ViT (backbone.model.model.visual.*)': len([k for k in hf_sd if 'model.model.visual' in k]),
    'LLM (backbone.model.model.language_model.*)': len([k for k in hf_sd if 'language_model' in k]),
    'VL self-attn (action_head.vl_self_attention.*)': len([k for k in hf_sd if 'vl_self_attention' in k]),
    'DiT (action_head.model.transformer_blocks.*)': len([k for k in hf_sd if 'transformer_blocks' in k]),
    'State/action CPU (action_head.state/action*)': len([k for k in hf_sd if 'state_encoder' in k or 'action_encoder' in k or 'action_decoder' in k]),
}
print('\nWeight groups:')
for name, count in subgraphs.items():
    print(f'  {name}: {count} tensors')

Checkpoint path: /home/ubuntu/.cache/huggingface/hub/models--nvidia--GR00T-N1.7-3B/snapshots/2fc962b973bccdd5d8ce4f67cc63b264d6886495
Safetensors shards: 2

Loading checkpoint...
  1031 tensors in 0.0s

Weight groups:
  ViT (backbone.model.model.visual.*): 315 tensors
  LLM (backbone.model.model.language_model.*): 178 tensors
  VL self-attn (action_head.vl_self_attention.*): 64 tensors
  DiT (action_head.model.transformer_blocks.*): 512 tensors
  State/action CPU (action_head.state/action*): 14 tensors


In [3]:
# Sample a few key tensor shapes
interesting_keys = [
    'backbone.model.model.visual.patch_embed.proj.weight',
    'backbone.model.model.language_model.layers.0.self_attn.q_proj.weight',
    'action_head.vl_self_attention.transformer_blocks.0.attn1.to_q.weight',
    'action_head.model.transformer_blocks.0.attn1.to_q.weight',
    'action_head.state_encoder.layer1.W',
]
print('Key tensor shapes:')
for k in interesting_keys:
    if k in hf_sd:
        print(f'  {k}: {tuple(hf_sd[k].shape)} {hf_sd[k].dtype}')
    else:
        print(f'  {k}: NOT FOUND')

Key tensor shapes:
  backbone.model.model.visual.patch_embed.proj.weight: (1024, 3, 2, 16, 16) torch.bfloat16
  backbone.model.model.language_model.layers.0.self_attn.q_proj.weight: (2048, 2048) torch.bfloat16
  action_head.vl_self_attention.transformer_blocks.0.attn1.to_q.weight: (2048, 2048) torch.bfloat16
  action_head.model.transformer_blocks.0.attn1.to_q.weight: (1536, 1536) torch.bfloat16
  action_head.state_encoder.layer1.W: (32, 132, 1024) torch.bfloat16


## 4. Compile Subgraphs (Skip-If-Compiled)

In [4]:
import os

COMPILED_DIR = os.path.join(PORT_DIR, 'compiled')

def is_compiled(name):
    return os.path.isfile(os.path.join(COMPILED_DIR, name, 'model.pt'))

print('Compilation status:')
for name in ['backbone', 'vl_self_attn', 'dit', 'vit']:
    status = 'compiled' if is_compiled(name) else 'NOT compiled'
    print(f'  {name:20s}: {status}')

Compilation status:
  backbone            : compiled
  vl_self_attn        : compiled
  dit                 : compiled
  vit                 : compiled


In [5]:
# Compile all TP=8 NEFFs if not already compiled
# This takes ~3 min (warm cache) or ~2 hours (cold cache)

needs_compile = not all(is_compiled(n) for n in ['backbone', 'vl_self_attn', 'dit'])
if needs_compile:
    print('Compiling backbone + vl_self_attn + dit...')
    os.chdir(PORT_DIR)
    !python compile_all.py 2>&1 | tail -20
else:
    print('All TP=8 NEFFs already compiled, skipping.')

if not is_compiled('vit'):
    print('Compiling ViT NEFF...')
    !python compile_vit.py 2>&1 | tail -10
else:
    print('ViT NEFF already compiled, skipping.')

All TP=8 NEFFs already compiled, skipping.
ViT NEFF already compiled, skipping.


## 5. Load Compiled NEFFs

In [6]:
from run_inference import load_model

print('Loading all compiled NEFFs + CPU weights...')
t0 = time.time()
model = load_model()
print(f'  Done in {time.time()-t0:.1f}s')

# Also load ViT NEFF
vit_neff_path = os.path.join(COMPILED_DIR, 'vit', 'model.pt')
if os.path.isfile(vit_neff_path):
    import torch_neuronx
    vit_neff = torch.jit.load(vit_neff_path)
    print(f'  ViT NEFF loaded from {vit_neff_path}')
else:
    vit_neff = None
    print('  ViT NEFF not found, will use CPU')

/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:100: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_baseline_shard_hidden: No module named 'neuronxcc.nki._private.blockwise_mm'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_bwd: No module named 'neuronxcc.nki._private.blockwise_mm_bwd'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_bwd_

Loading all compiled NEFFs + CPU weights...
Loading checkpoint...
  Loaded 1031 tensors
Loading CPU weights...
Loading backbone subgraph...


Loading backbone NEFF from /home/ubuntu/groot-n1-port/compiled/backbone


2026-May-05 01:54:33.0005 6624:6902 [2] int nccl_net_ofi_create_plugin(nccl_net_ofi_plugin_t**):213 CCOM WARN NET/OFI Failed to initialize sendrecv protocol
2026-May-05 01:54:33.0011 6624:6902 [2] int nccl_net_ofi_create_plugin(nccl_net_ofi_plugin_t**):354 CCOM WARN NET/OFI aws-ofi-nccl initialization failed
2026-May-05 01:54:33.0016 6624:6902 [2] ncclResult_t nccl_net_ofi_init_no_atexit_fini_v6(ncclDebugLogger_t):183 CCOM WARN NET/OFI Initializing plugin failed
2026-May-05 01:54:33.0022 6624:6902 [2] net_plugin.cc:97 CCOM WARN OFI plugin initNet() failed is EFA enabled?


Loading VL self-attention subgraph...
Loading VL self-attn NEFF from /home/ubuntu/groot-n1-port/compiled/vl_self_attn


Loading DiT subgraph...
Loading DiT NEFF from /home/ubuntu/groot-n1-port/compiled/dit


Model loaded.
  Done in 9.7s
  ViT NEFF loaded from /home/ubuntu/groot-n1-port/compiled/vit/model.pt


## 6. Sample Inference with Dummy Inputs

In [7]:
from vlm_backbone_block import run_vit_cpu, make_backbone_inputs
from config_constants import BATCH_SIZE, MAX_STATE_DIM

hf_sd = model._hf_sd

print('Step 1: ViT preprocessing (CPU)...')
backbone_in = make_backbone_inputs(B=BATCH_SIZE, n_text=16)
t0 = time.time()
inputs_embeds, pre_cos, pre_sin = run_vit_cpu(
    backbone_in['input_ids'], backbone_in['attention_mask'],
    backbone_in['pixel_values'], backbone_in['image_grid_thw'],
    hf_sd=hf_sd,
)
print(f'  Done in {(time.time()-t0)*1000:.1f}ms')
print(f'  inputs_embeds: {inputs_embeds.shape}, pre_cos: {pre_cos.shape}')

state = torch.zeros(BATCH_SIZE, 1, MAX_STATE_DIM, dtype=torch.bfloat16)
embodiment_id = torch.zeros(BATCH_SIZE, dtype=torch.long)

Step 1: ViT preprocessing (CPU)...


  Done in 22549.5ms
  inputs_embeds: torch.Size([1, 256, 2048]), pre_cos: torch.Size([1, 256, 128])


In [8]:
print('Step 2: Full inference (backbone + VL-SA + DiT × 4)...')
# Warmup
with torch.no_grad():
    _ = model.generate_actions(inputs_embeds, pre_cos, pre_sin, state, embodiment_id)

t0 = time.time()
with torch.no_grad():
    actions = model.generate_actions(
        inputs_embeds, pre_cos, pre_sin, state, embodiment_id,
        profile=True,   # prints per-subgraph timing
    )
elapsed = (time.time() - t0) * 1000

print(f'\nTotal inference: {elapsed:.1f}ms')
print(f'Actions shape:   {actions.shape}')
print(f'Actions mean:    {actions.float().mean().item():.4f}')
print(f'Actions std:     {actions.float().std().item():.4f}')
assert not torch.isnan(actions).any(), 'NaN in actions!'
print('No NaN in output. OK.')

Step 2: Full inference (backbone + VL-SA + DiT × 4)...
  [profile] backbone NEFF:    10.63ms
  [profile] VL self-attn NEFF:7.05ms
  [profile] DiT step 1/4:   9.53ms
  [profile] DiT step 2/4:   6.10ms
  [profile] DiT step 3/4:   6.08ms
  [profile] DiT step 4/4:   6.09ms
  [profile] DiT total:        27.80ms

Total inference: 56.4ms
Actions shape:   torch.Size([1, 40, 132])
Actions mean:    0.0090
Actions std:     0.9983
No NaN in output. OK.


## 7. Benchmark

In [9]:
import numpy as np

NUM_WARMUP = 5
NUM_MEASURE = 20

print(f'Benchmark: {NUM_WARMUP} warmup + {NUM_MEASURE} measure iterations')
latencies = []

for i in range(NUM_WARMUP + NUM_MEASURE):
    t0 = time.perf_counter()
    with torch.no_grad():
        actions = model.generate_actions(inputs_embeds, pre_cos, pre_sin, state, embodiment_id)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    if i >= NUM_WARMUP:
        latencies.append(elapsed_ms)

lats = np.array(latencies)
print(f'\nResults (excl. ViT):')
print(f'  Mean:       {lats.mean():.2f} ms')
print(f'  Median:     {np.median(lats):.2f} ms')
print(f'  P95:        {np.percentile(lats, 95):.2f} ms')
print(f'  Throughput: {1000/lats.mean():.1f} inferences/sec')

Benchmark: 5 warmup + 20 measure iterations



Results (excl. ViT):
  Mean:       43.19 ms
  Median:     42.94 ms
  P95:        45.46 ms
  Throughput: 23.2 inferences/sec


In [10]:
# ViT NEFF vs CPU comparison
from config_constants import VIT_PATCH_SIZE, VIT_TEMPORAL_PATCH_SIZE

NPATCH = 256
pv_flat = torch.randn(
    NPATCH, 3 * VIT_TEMPORAL_PATCH_SIZE * VIT_PATCH_SIZE * VIT_PATCH_SIZE,
    dtype=torch.bfloat16,
)

# ViT CPU timing
cpu_lats = []
for _ in range(5):
    t0 = time.perf_counter()
    run_vit_cpu(
        backbone_in['input_ids'], backbone_in['attention_mask'],
        backbone_in['pixel_values'], backbone_in['image_grid_thw'],
        hf_sd=hf_sd,
    )
    cpu_lats.append((time.perf_counter() - t0) * 1000)
print(f'ViT CPU: mean={np.mean(cpu_lats):.1f}ms')

# ViT NEFF timing (if available)
if vit_neff is not None:
    neff_lats = []
    for _ in range(10 + 20):  # 10 warmup, 20 measure
        t0 = time.perf_counter()
        _ = vit_neff(pv_flat)
        neff_lats.append((time.perf_counter() - t0) * 1000)
    neff_lats = neff_lats[10:]
    print(f'ViT NEFF: mean={np.mean(neff_lats):.2f}ms  (speedup: {np.mean(cpu_lats)/np.mean(neff_lats):.0f}x)')

ViT CPU: mean=331.3ms
ViT NEFF: mean=4.95ms  (speedup: 67x)


## 8. Correctness Validation

In [11]:
# Run validate_neffs.py and capture output
import subprocess
result = subprocess.run(
    ['python', 'validate_neffs.py'],
    cwd=PORT_DIR,
    capture_output=True, text=True, timeout=900,
)
# Print stdout; filter out neuron/torch warnings from stderr
for line in result.stdout.split("
"):
    print(line)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])


Loading HF checkpoint...
  1031 tensors in 0.0s

TEST 1: VL Self-Attention — NEFF vs SelfAttentionTransformer
  NEFF loaded.
Total number of SelfAttentionTransformer parameters:  201433088
  Reference loaded.
  shapes: ref=torch.Size([1, 256, 2048])  neff=torch.Size([1, 256, 2048])
  mean_diff=0.00667  max_diff=0.2500  cos_sim=0.999941  → PASS ✓

TEST 2: DiT Action Head — NEFF vs AlternateVLDiT
  NOTE: deviation expected — NEFF attends all conditioning tokens per block;
  AlternateVLDiT alternates image/text tokens by block. cos_sim threshold relaxed to 0.95.
  NEFF loaded.
Total number of DiT parameters:  1091722240
  Reference loaded.
  shapes: ref=torch.Size([1, 41, 1024])  neff=torch.Size([1, 41, 1024])
  mean_diff=0.00153  max_diff=0.0182  cos_sim=0.999930  → PASS ✓

TEST 3: Backbone LLM (16 layers) — NEFF vs Qwen3VL language_model
  NEFF loaded.
  Reference loaded (16-layer Qwen3VL).
  Running ViT on CPU...
  inputs_embeds: torch.Size([1, 256, 2048]), pre_cos: torch.Size([1, 256,

In [12]:
# Run open-loop eval
import subprocess
result = subprocess.run(
    ['python', 'open_loop_eval.py'],
    cwd=PORT_DIR,
    capture_output=True, text=True, timeout=900,
)
for line in result.stdout.split("
"):
    print(line)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])


GR00T N1.7 Open-Loop Evaluation (Dummy Inputs)
Neuron pipeline vs HF reference

Note: This cell runs open_loop_eval.py as a subprocess. Due to NeuronCore
allocation by the notebook kernel, run it standalone for full output:
  python open_loop_eval.py

Expected results (from standalone run, see STATUS.md):
  Open-loop MSE within 10% of HF reference for all 3 trajectories.
  See STATUS.md Phase 5 for detailed results.
